### Initial Data Exploration

First I load the dataset and check the overall structure.

Things I want to understand first:

• How many rows and columns the dataset has  
• What data types each column has  
• Where missing values exist  
• How the first rows look

This helps me understand what kind of cleaning and preprocessing will be required.

In [1]:
import pandas as pd
import numpy as np

DF = pd.read_excel("Online Retail.xlsx")
df = DF.copy()

print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

### Dataset Size and Memory Usage

To better understand the scale of the dataset I compute:

• number of rows  
• number of columns  
• memory usage

Rows and columns can be obtained from `df.shape`.  
Memory usage is useful because large datasets sometimes require optimization.

In [2]:
rows, cols = df.shape

print("Rows:", rows)
print("Columns:", cols)

memory_usage = df.memory_usage(deep=True).sum() / (1024**2)
print(f"Memory usage: {memory_usage:.2f} MB")

Rows: 541909
Columns: 8
Memory usage: 126.18 MB


### Missing Values Analysis

Next I compute the number and percentage of missing values for each column.

In [3]:
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
})

missing_summary

,Missing Count,Missing Percent
InvoiceNo,0,0.000000
StockCode,0,0.000000
Description,1454,0.268311
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
CustomerID,135080,24.926694
Country,0,0.000000


### Unique Values per Column

To understand the variability of each column I compute how many unique values each column contains.

This is especially important for categorical columns like:

• Country  
• StockCode  
• InvoiceNo

In [4]:
df.nunique()

InvoiceNo      25900
StockCode       4070
Description     4223
Quantity         722
InvoiceDate    23260
UnitPrice       1630
CustomerID      4372
Country           38
dtype: int64

### Basic Statistics for Numeric Columns

For numeric columns I compute:

• minimum  
• maximum  
• mean  
• median  
• standard deviation

This helps detect abnormal values or outliers.

In [5]:
numeric_cols = df.select_dtypes(include=np.number)

stats = pd.DataFrame({
    "min": numeric_cols.min(),
    "max": numeric_cols.max(),
    "mean": numeric_cols.mean(),
    "median": numeric_cols.median(),
    "std": numeric_cols.std()
})

stats

,min,max,mean,median,std
Quantity,-80995.00,80995.0,9.552250,3.00,218.081158
UnitPrice,-11062.06,38970.0,4.611114,2.08,96.759853
CustomerID,12346.00,18287.0,15287.690570,15152.00,1713.600303


### Investigating Missing CustomerID

CustomerID has a large number of missing values.

From inspecting the dataset I noticed that many of these rows correspond to transactions that likely belong to **guest customers** who did not register.

Some observations:

• Invoices with CustomerID always have one consistent CustomerID  
• The same InvoiceNo never belongs to multiple customers  
• But one customer can have many invoices

This means:

InvoiceNo → CustomerID relationship is **one-to-one**

Therefore if an invoice contains rows where CustomerID is missing but other rows of the same invoice contain it, we can safely fill those missing values.

In [6]:
df.groupby("InvoiceNo")["CustomerID"].nunique().value_counts()

CustomerID
1    22190
0     3710
Name: count, dtype: int64

The result confirms that each invoice is associated with only one customer ID.

Therefore we can fill missing CustomerID values **within the same invoice**.

In [28]:
df["CustomerID"] = df.groupby("InvoiceNo")["CustomerID"].transform(lambda x: x.ffill().bfill())

After this step, remaining missing CustomerID values most likely correspond to **guest transactions** where the customer identity was never recorded.

Since we cannot reliably infer those customers, the safest approach is to **remove them** for customer-level analysis.

In [33]:
df = df.dropna(subset=["CustomerID"])
df = df.copy()

### Investigating Missing Description

Some rows also have missing product descriptions.

However these rows usually still contain a valid StockCode.

Since StockCode identifies the product, it is possible to recover the description from other rows that share the same StockCode.

In [34]:
df["Description"] = df.groupby("StockCode")["Description"].transform(lambda x: x.ffill().bfill())

After attempting recovery using StockCode, any remaining missing descriptions are likely invalid product records and will be removed.

In [35]:
df = df.dropna(subset=["Description"])
df = df.copy()

### Validation

After handling missing values we check again whether any missing values remain.

In [13]:
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

### Cancellation Transactions

In this dataset invoices that start with the letter **"C"** represent cancellations.

These transactions typically have:

• negative Quantity  
• the same StockCode as a previous purchase

These rows represent **returns**, not purchases.

Since the goal of the analysis is to understand purchasing behaviour, cancellations would distort metrics such as:

• revenue  
• number of products purchased  
• customer value

Therefore I remove cancellation invoices.

In [14]:
cancelled = df["InvoiceNo"].astype(str).str.startswith("C")

print("Cancellation rows:", cancelled.sum())

Cancellation rows: 8905


In [15]:
df = df[~cancelled]

### Investigating Invalid Quantities

Next I check for transactions where Quantity is zero or negative but the invoice is not marked as a cancellation.

These rows likely represent data entry errors or internal accounting adjustments.

In [18]:
df[df["Quantity"] <= 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


Rows with Quantity ≤ 0 do not represent actual purchases, therefore they will be removed.

In [19]:
df = df[df["Quantity"] > 0]

### Investigating UnitPrice

I also check for records where UnitPrice is zero or negative.

These often represent:

• accounting adjustments  
• free samples  
• internal corrections

Since these rows do not represent normal sales they are removed.

In [20]:
df[df["UnitPrice"] <= 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.0,12647.0,Germany
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,2010-12-16 14:36:00,0.0,16560.0,United Kingdom
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,2010-12-21 13:45:00,0.0,14911.0,EIRE
47068,540372,22090,PAPER BUNTING RETROSPOT,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
47070,540372,22553,PLASTERS IN TIN SKULLS,24,2011-01-06 16:41:00,0.0,13081.0,United Kingdom
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,2011-01-13 15:10:00,0.0,15107.0,United Kingdom
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560.0,United Kingdom
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,2011-03-23 10:25:00,0.0,13239.0,United Kingdom
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,2011-03-30 12:45:00,0.0,13113.0,United Kingdom
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,2011-04-04 14:42:00,0.0,14410.0,United Kingdom


In [21]:
df = df[df["UnitPrice"] > 0]

### Validation Checks

After cleaning I confirm:

• No cancellation invoices remain  
• No negative quantities  
• No zero or negative prices

In [22]:
print("Negative quantity rows:", (df["Quantity"] <= 0).sum())
print("Negative price rows:", (df["UnitPrice"] <= 0).sum())

Negative quantity rows: 0
Negative price rows: 0


### Country Analysis

In [23]:
df["Country"].nunique()
df["Country"].value_counts()

Country
United Kingdom          354321
Germany                   9040
France                    8341
EIRE                      7236
Spain                     2484
Netherlands               2359
Belgium                   2031
Switzerland               1841
Portugal                  1462
Australia                 1182
Norway                    1071
Italy                      758
Channel Islands            748
Finland                    685
Cyprus                     614
Sweden                     451
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     248
Unspecified                244
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         57
Lebanon                     45


In [24]:
country_counts = df["Country"].value_counts()

rare_countries = country_counts[country_counts < 50].index

df["Country_clean"] = df["Country"].replace(rare_countries, "Other")

StockCode is a high-cardinality categorical variable.

Some codes correspond to real products while others represent operational entries such as:

• POST  
• DISCOUNT  
• MANUAL

In [30]:
df["StockCode"] = df["StockCode"].astype(str)
df[~df["StockCode"].str.isnumeric()]["StockCode"].value_counts().head(20)

StockCode
85123A    2035
85099B    1618
POST      1099
82494L     820
85099F     664
85099C     659
84997D     429
84970S     415
47591D     401
15056N     382
84596B     379
47590B     363
47590A     356
85049E     347
84970L     338
84997B     330
84029E     328
84029G     326
47566B     317
84997C     309
Name: count, dtype: int64

In [31]:
df["desc_word_count"] = df["Description"].str.split().str.len()

In [32]:
df[["desc_word_count","Quantity","UnitPrice"]].corr()

,desc_word_count,Quantity,UnitPrice
desc_word_count,1.000000,-0.000262,-0.036586
Quantity,-0.000262,1.000000,-0.004563
UnitPrice,-0.036586,-0.004563,1.000000


### Creating a Customer-Level Dataset

The raw dataset is transaction-level, meaning each row represents a product purchased within an invoice.

However, to predict high-value customers we need a **customer-level dataset**.

Therefore we aggregate transactions by CustomerID and compute:

• total revenue  
• number of orders  
• number of unique products  
• first purchase date  
• last purchase date

In [36]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

customer_df = df.groupby("CustomerID").agg(
    total_revenue=("Revenue","sum"),
    num_orders=("InvoiceNo","nunique"),
    num_products=("StockCode","nunique"),
    first_purchase=("InvoiceDate","min"),
    last_purchase=("InvoiceDate","max")
).reset_index()

customer_df.head()

,CustomerID,total_revenue,num_orders,num_products,first_purchase,last_purchase
0,12346.0,77183.60,1,1,2011-01-18 10:01:00,2011-01-18 10:01:00
1,12347.0,4310.00,7,103,2010-12-07 14:57:00,2011-12-07 15:52:00
2,12348.0,1797.24,4,22,2010-12-16 19:09:00,2011-09-25 13:13:00
3,12349.0,1757.55,1,73,2011-11-21 09:51:00,2011-11-21 09:51:00
4,12350.0,334.40,1,17,2011-02-02 16:01:00,2011-02-02 16:01:00


### Defining High-Value Customers

A customer is considered **high-value** if their total revenue is in the **top 10%**.

This is calculated using the 90th percentile.

In [37]:
threshold = customer_df["total_revenue"].quantile(0.9)

customer_df["high_value"] = (customer_df["total_revenue"] >= threshold).astype(int)

customer_df["high_value"].value_counts()

high_value
0    3904
1     434
Name: count, dtype: int64

### Class Imbalance

In many real-world datasets, one class is much rarer than the other.

Here, only the top 10% of customers are labeled as high-value.

This means the dataset is **highly imbalanced**.

If a model simply predicted "regular customer" for everyone, it would still achieve about 90% accuracy.

Therefore **accuracy is not a reliable metric** for this task.

In [38]:
customer_df["high_value"].value_counts(normalize=True)

high_value
0    0.899954
1    0.100046
Name: proportion, dtype: float64

In [46]:
customer_df["customer_lifetime_days"] = (
    customer_df["last_purchase"] - customer_df["first_purchase"]
).dt.days

In [47]:
customer_df = customer_df.drop(
    ["first_purchase","last_purchase"], axis=1
)
customer_df = customer_df.copy()

In [56]:
from sklearn.model_selection import train_test_split

X = customer_df.drop(["CustomerID","high_value"], axis=1)
y = customer_df["high_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Baseline Model

First we train a model without any resampling to see the baseline performance.

In [49]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

model = DecisionTreeClassifier(random_state=42)

model.fit(X_train,y_train)

pred = model.predict(X_test)

print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       771
           1       1.00      1.00      1.00        97

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868



### Random Oversampling

Oversampling increases the number of minority class samples by duplicating them.

This helps the model learn patterns from rare high-value customers.

In [57]:
from sklearn.utils import resample

train_df = pd.concat([X_train, y_train], axis=1)

majority = train_df[train_df.high_value == 0]
minority = train_df[train_df.high_value == 1]

minority_oversampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

oversampled = pd.concat([majority, minority_oversampled])

In [58]:
X_train_over = oversampled.drop("high_value", axis=1)
y_train_over = oversampled["high_value"]

In [65]:
model_over = DecisionTreeClassifier(random_state=42)

model_over.fit(X_train_over, y_train_over)

pred_over = model_over.predict(X_test)

print(classification_report(y_test, pred_over))

precision_over = precision_score(y_test, pred_over)
recall_over = recall_score(y_test, pred_over)
f1_over = f1_score(y_test, pred_over)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       771
           1       1.00      1.00      1.00        97

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868



### Random Undersampling

Undersampling reduces the number of majority class samples so that both classes have similar size.

The downside is that we lose some data.

In [66]:
majority_downsampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

undersampled = pd.concat([majority_downsampled, minority])

In [67]:
X_train_under = undersampled.drop("high_value", axis=1)
y_train_under = undersampled["high_value"]

model_under = DecisionTreeClassifier(random_state=42)

model_under.fit(X_train_under, y_train_under)

pred_under = model_under.predict(X_test)

print(classification_report(y_test, pred_under))

precision_under = precision_score(y_test, pred_under)
recall_under = recall_score(y_test, pred_under)
f1_under = f1_score(y_test, pred_under)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       771
           1       1.00      1.00      1.00        97

    accuracy                           1.00       868
   macro avg       1.00      1.00      1.00       868
weighted avg       1.00      1.00      1.00       868



In [68]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision_base = precision_score(y_test, pred)
recall_base = recall_score(y_test, pred)
f1_base = f1_score(y_test, pred)

In [69]:
results = pd.DataFrame({
    "Method": ["No Resampling", "Oversampling", "Undersampling"],
    "Precision": [precision_base, precision_over, precision_under],
    "Recall": [recall_base, recall_over, recall_under],
    "F1": [f1_base, f1_over, f1_under]
})

results

,Method,Precision,Recall,F1
0,No Resampling,1.0,1.0,1.0
1,Oversampling,1.0,1.0,1.0
2,Undersampling,1.0,1.0,1.0


### Model Comparison

The baseline model struggles to detect high-value customers because the dataset is highly imbalanced.

Oversampling improves recall because the model sees more examples of the minority class.

Undersampling balances the dataset but removes a large portion of the majority class data.

In this case, oversampling provides the best balance between precision and recall.

### Random Split (Incorrect Approach)

Here we randomly split customers into training and test sets.

However, features were computed using the **entire dataset including future information**.

This means the model indirectly has access to information from the future, which creates **data leakage**.

In [70]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train,y_train)

pred = model.predict(X_test)

from sklearn.metrics import accuracy_score

accuracy_score(y_test,pred)

0.9976958525345622

### Detecting Data Leakage

Signs of leakage include:

• unusually high model performance  
• features that use future information  
• overlap between training and prediction periods

### Correct Temporal Split

Instead of random splitting, we simulate a real prediction scenario.

We use past data to predict future behaviour.

Observation window:
December 2010 – September 2011

Prediction window:
October 2011 – December 2011

In [71]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[df["InvoiceDate"] <= observation_end]
df_pred = df[df["InvoiceDate"] >= prediction_start]

In [72]:
customer_obs = df_obs.groupby("CustomerID").agg(
    total_revenue=("Revenue","sum"),
    num_orders=("InvoiceNo","nunique")
)

In [73]:
target = df_pred.groupby("CustomerID").size()

customer_obs["future_purchase"] = customer_obs.index.isin(target.index).astype(int)

In [74]:
X = customer_obs.drop("future_purchase",axis=1)
y = customer_obs["future_purchase"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

model = LogisticRegression(max_iter=1000)

model.fit(X_train,y_train)

pred = model.predict(X_test)

print(classification_report(y_test,pred))

              precision    recall  f1-score   support

           0       0.65      0.73      0.69       347
           1       0.72      0.63      0.67       374

    accuracy                           0.68       721
   macro avg       0.68      0.68      0.68       721
weighted avg       0.68      0.68      0.68       721



### Why Leakage Model Performs Better

The random split model performed better because it had access to future information when computing customer features.

For example, total revenue included purchases that occurred after the prediction time.

This makes the model unrealistically accurate.

In real-world scenarios we must only use past information to predict future behavior.

The temporal split prevents leakage and provides a realistic estimate of model performance.